# MultiHb — multimodal Hb estimation (eye image + height / weight / BMI)

Reference implementation for the paper. This notebook runs exactly the configuration the
paper reports: regression only, three tabular variables (height, weight, BMI), fourteen
backbones, four fusion operators, and a three-stage study under one leakage-controlled
protocol.

The tabular-only reference is **not** reported.
`multimamba.ipynb` is kept in this repository as the original exploratory notebook.

## Section 1 — Configuration (edit only this block)

In [ ]:
# ==============================================================
#            ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# ---- Cross-validation --------------------------------------------------------
K = 5                       # stratified five-fold CV on the train pool
assert 1 <= K <= 5, "K must be between 1 and 5"

# ---- Task --------------------------------------------------------------------
TASK = "regression"     # "classification" (anemic < HB_THRESH) | "regression" (exact Hb)

# ---- Dataset paths -----------------------------------------------------------
IMAGE_DIR   = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/left_eye"
CSV_PATH    = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/merge_excel_1.xlsx"
IMAGE_DIR_2 = ""            # optional 2nd dataset slot (e.g. right_eye)
CSV_PATH_2  = ""

# ---- Column names (must match the Excel headers EXACTLY) ---------------------
IMAGE_COL  = "Patient ID"
HB_COL     = "Haemoglobin (gm/dL)"
HB_THRESH  = 12.0
HEIGHT_COL = "Height"
WEIGHT_COL = "Weight"

# ---- Multimodal / tabular ----------------------------------------------------
USE_TABULAR      = True                    # master switch: add tabular vars to the model
BMI_COL          = "BMI"                   # derived below from height and weight
NUMERIC_FEATURES = [HEIGHT_COL, WEIGHT_COL, BMI_COL]   # standardised tabular features
FUSION           = "attn"                  # MAIN-pass fusion: "concat"|"film"|"gate"|"attn"
FUSION_MODES     = ["concat", "film", "gate", "attn"]   # compared in the ablation
TAB_HIDDEN       = 64                       # tabular MLP width
FUSE_DIM         = 128                      # token dim for "attn" fusion
TAB_DROPOUT      = 0.2

# ---- HB range filter / augmentation -----------------------------------------
HB_FILTER_MIN = 7.0
HB_FILTER_MAX = 15.0
USE_AUGMENTATION = True
AUG_BINS         = 10

# ---- Training ----------------------------------------------------------------
EPOCHS      = 40
BATCH_SIZE  = 32
LR          = 1e-4
VAL_SPLIT   = 0.2          # used when K == 1
TEST_SPLIT  = 0.1          # fixed held-out test set (locked across all folds)
NUM_WORKERS = 4
SEED        = 42
EARLY_STOP  = 8
SCHEDULER   = "cosine"     # "cosine" | "plateau" | None
GRAD_ACCUM  = 1
WARMUP_EPOCHS = 3
USE_AMP     = True
FREEZE_BACKBONE = True     # EfficientMamba / timm baselines
IMG_SIZE    = 224

# ---- Preprocessing -----------------------------------------------------------
PREPROCESS = {"colorspace": "RGB", "use_clahe": False,
              "clahe_clip": 2.0, "clahe_grid": (8, 8)}

# ---- Classification-specific settings ---------------------------------------
CLS_CONFIG = {"use_focal_loss": True, "focal_gamma": 2.0,
              "use_class_weights": True, "threshold": 0.5}

# ---- Regression-specific settings -------------------------------------------
REG_CONFIG = {"normalize_hb": True, "hb_min": HB_FILTER_MIN, "hb_max": HB_FILTER_MAX,
              "loss_fn": "huber", "huber_delta": 1.0}

# ---- Novel-model architecture cfg -------------------------------------------
NEW_MODEL_CFG = {"embed_dim": 128, "depth": 4, "d_state": 64,
                 "colour_proj": "learned", "n_experts": 5}

# ---- Which models to run in the MAIN pass -----------------------------------
RUN = {
    # Mamba variants
    "Mamba1": True, "Mamba2": True, "Mamba3": True, "VMamba": True,
    "MambaVision": True, "MedMamba": True,
    "EfficientMamba": True, "AdaptiveScan": True, "CrossFusion": True,
    # VSSD, DSA_Mamba and MoEMamba are present in the registry but not reported here
    # CNN / ViT baselines
    "ResNet50": True, "EfficientNetB3": True, "DenseNet121": True,
    "MobileNetV3": True, "ViT_B16": True,
}

# ---- Ablation config (runs at the END of the notebook) ----------------------
ABLATION_MODEL        = "ViT_B16"  # backbone carried into Stages 2 and 3
RUN_ABLATION_FUSION   = True       # compare every mode in FUSION_MODES
RUN_ABLATION_VARIABLE = True       # image-only -> +Height -> +Weight -> +BMI -> all three
RUN_ABLATION_TABONLY  = False      # not reported in this study

# Each step = (label, numeric_feature_list).  Shows each variable's marginal gain.
ABLATION_STEPS = [
    ("image-only",          []),
    ("+Height",             [HEIGHT_COL]),
    ("+Weight",             [WEIGHT_COL]),
    ("+BMI",                [BMI_COL]),
    ("+Height+Weight+BMI",  [HEIGHT_COL, WEIGHT_COL, BMI_COL]),
]

# ---- Repo path autodetect ----------------------------------------------------
import os, sys
def _find_repo_root():
    cands = [os.getcwd(), os.path.join(os.getcwd(), "ALLMAMBA")]
    try:
        for d in os.listdir(os.getcwd()):
            p = os.path.join(os.getcwd(), d)
            if os.path.isdir(p): cands.append(p)
    except Exception: pass
    cands.append("/home/junaid/Desktop/MULTIMAMBA")
    for c in cands:
        if os.path.isdir(os.path.join(c, "MAMBA_MODELS")): return c
    return os.getcwd()

REPO_ROOT  = _find_repo_root()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)

CV_DESC = "single holdout" if K == 1 else f"{K}-fold StratifiedKFold"
print(f"TASK        : {TASK}")
print(f"Cross-val   : K={K}  ({CV_DESC})")
print(f"Multimodal  : USE_TABULAR={USE_TABULAR}  fusion={FUSION}")
print(f"Numeric feat: {NUMERIC_FEATURES}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"Models ON   : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install dependencies

In [ ]:
# ---- Install deps (Kaggle). Comment out if already installed. ----
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
# _pip("mamba-ssm", "causal-conv1d", "einops", "timm", "pandas", "openpyxl",
#      "scikit-learn", "tqdm", "matplotlib", "opencv-python")
print("If packages are missing, uncomment the _pip(...) line above and re-run.")


## Section 3 — Repo path

In [ ]:
# ── Auto clone-or-update the repo, then make it importable ────────────────────
# On Kaggle: set REPO_URL to YOUR repo and leave AUTO_CLONE=True -> it clones once,
#            and on later runs it just pulls the latest.
# On your local machine (repo already in the folder): set AUTO_CLONE=False.
import subprocess, os, sys

REPO_URL   = "https://github.com/junaidmaqbool/MULTIMAMBA.git"   # <-- change to match your repo EXACTLY
CLONE_DIR  = "/home/junaid/Desktop/MULTIMAMBA"
AUTO_CLONE = True

if AUTO_CLONE and not os.path.isdir(MODELS_DIR):
    if os.path.isdir(os.path.join(CLONE_DIR, "MAMBA_MODELS")):
        print("Updating existing clone ..."); subprocess.run(["git","-C",CLONE_DIR,"pull","--ff-only"], check=False)
    else:
        print("Cloning", REPO_URL); subprocess.run(["git","clone","--depth","1",REPO_URL,CLONE_DIR], check=False)
    REPO_ROOT  = _find_repo_root()
    MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
    if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)

print("Repo present:", os.path.isdir(MODELS_DIR), "|  MODELS_DIR:", MODELS_DIR)

## Section 4 — Imports and device

In [ ]:
import os, sys, math, time, warnings, traceback, importlib.util, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report, roc_auc_score,
                             mean_absolute_error, f1_score, confusion_matrix, r2_score)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}   PyTorch: {torch.__version__}")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)


## Section 5 — Data: merge, derive BMI, lock the test set

In [ ]:
from common.augment import merge_and_filter_datasets, make_balanced_loader

# ---------- image preprocessing (same as run_ensemble) ----------
class CLAHETransform:
    def __call__(self, img):
        import cv2
        lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=PREPROCESS["clahe_clip"],
                                tileGridSize=PREPROCESS["clahe_grid"])
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

class ToLABTensor:
    def __call__(self, img):
        import cv2
        lab = cv2.cvtColor(np.array(img).astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[:, :, 0] /= 255.0
        lab[:, :, 1] = (lab[:, :, 1] - 128.0) / 127.0
        lab[:, :, 2] = (lab[:, :, 2] - 128.0) / 127.0
        return torch.from_numpy(lab.transpose(2, 0, 1))

class ToHSVTensor:
    def __call__(self, img):
        import cv2
        hsv = cv2.cvtColor(np.array(img).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] /= 179.0; hsv[:, :, 1] /= 255.0; hsv[:, :, 2] /= 255.0
        return torch.from_numpy(hsv.transpose(2, 0, 1))

def build_transforms(augment):
    steps = []
    if augment:
        steps += [transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
                  transforms.RandomCrop(IMG_SIZE),
                  transforms.RandomHorizontalFlip(),
                  transforms.RandomRotation(15),
                  transforms.ColorJitter(0.25, 0.25, 0.15)]
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
    if PREPROCESS["use_clahe"]:
        steps.append(CLAHETransform())
    cs = PREPROCESS["colorspace"].upper()
    if cs == "RGB":
        steps += [transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)]
    elif cs == "CIELAB": steps.append(ToLABTensor())
    elif cs == "HSV":    steps.append(ToHSVTensor())
    elif cs in ("GRAY", "GRAYSCALE"):
        steps += [transforms.Grayscale(3), transforms.ToTensor(),
                  transforms.Normalize((0.5,)*3, (0.5,)*3)]
    else: raise ValueError(f"Unknown colorspace {PREPROCESS['colorspace']}")
    return transforms.Compose(steps)

_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]
def find_image_path(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

# ---------- load + merge ----------
def _load_full_df():
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p) if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df); slot_dirs.append(img_d)
    if not slot_dfs:
        raise RuntimeError("No dataset slots active. Set CSV_PATH and IMAGE_DIR.")
    df = merge_and_filter_datasets(
        dataframes=slot_dfs, image_dirs=slot_dirs, hb_col=HB_COL, image_col=IMAGE_COL,
        hb_filter_min=HB_FILTER_MIN, hb_filter_max=HB_FILTER_MAX,
        find_image_path_fn=find_image_path, verbose=True)
    if len(df) == 0:
        raise RuntimeError("Zero valid samples after filtering.")
    return df.reset_index(drop=True)

FULL = _load_full_df()

# ---------- validate tabular columns ----------
def _report_col(col):
    if col not in FULL.columns:
        print(f"  [WARN] tabular column '{col}' NOT found in the sheet -> "
              f"it will be imputed as 0. Available: {list(FULL.columns)}")
        return False
    n_ok = pd.to_numeric(FULL[col], errors='coerce').notna().sum()
    print(f"  [OK]  '{col}': {n_ok}/{len(FULL)} usable values")
    return True

def add_bmi_column(df, height_col=HEIGHT_COL, weight_col=WEIGHT_COL, out_col=BMI_COL):
    """BMI = weight(kg) / height(m)^2.  Auto-detects whether height is in cm or m."""
    if df is None or height_col not in df.columns or weight_col not in df.columns:
        return df
    h = pd.to_numeric(df[height_col], errors="coerce")
    w = pd.to_numeric(df[weight_col], errors="coerce")
    med = h.median(skipna=True)
    h_m = h / 100.0 if (pd.notna(med) and med > 3) else h          # >3 => given in cm
    df[out_col] = (w / (h_m ** 2)).replace([np.inf, -np.inf], np.nan)
    return df

FULL = add_bmi_column(FULL)

print("Tabular column check:")
for c in NUMERIC_FEATURES: _report_col(c)

# ---------- fixed test split + train/val pool ----------
_binary = (FULL[HB_COL] < HB_THRESH).astype(int)
POOL_DF, TEST_DF = train_test_split(
    FULL, test_size=TEST_SPLIT, random_state=SEED, stratify=_binary)
# NOTE: each subject contributes exactly one left-eye image, so splitting rows is
# splitting subjects; no subject appears in more than one partition.
POOL_DF  = POOL_DF.reset_index(drop=True)
TEST_DF  = TEST_DF.reset_index(drop=True)
POOL_LBL = (POOL_DF[HB_COL] < HB_THRESH).astype(int).values

n_anem = int(_binary.sum())
print(f"\n  Samples: {len(FULL)}  | Anemic(<{HB_THRESH}): {n_anem} "
      f"({n_anem/len(FULL)*100:.1f}%)  Normal: {len(FULL)-n_anem}")
print(f"  Train+Val pool: {len(POOL_DF)}   Fixed Test: {len(TEST_DF)}")
print(f"  Hb -> mean={FULL[HB_COL].mean():.2f}  std={FULL[HB_COL].std():.2f}  "
      f"min={FULL[HB_COL].min():.2f}  max={FULL[HB_COL].max():.2f}")


## Section 6 — Tabular preprocessing and multimodal dataset

In [ ]:
def fit_tab_stats(df, numeric_feats):
    """Per-fold standardisation stats from TRAIN rows only. Returns {col:(mean,std)}."""
    stats = {}
    for c in numeric_feats:
        if c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce")
            m = float(v.mean()) if v.notna().any() else 0.0
            s = float(v.std())  if v.notna().any() else 1.0
        else:
            m, s = 0.0, 1.0
        stats[c] = (m, max(s, 1e-6))
    return stats

def _row_tab_vector(row, numeric_feats, stats):
    vec = []
    for c in numeric_feats:
        m, s = stats[c]
        x = pd.to_numeric(pd.Series([row.get(c, np.nan)]), errors="coerce").iloc[0]
        vec.append(0.0 if pd.isna(x) else (float(x) - m) / s)   # nan -> mean(=0)
    return np.asarray(vec, dtype=np.float32)

class MMDataset(Dataset):
    def __init__(self, df, transform, numeric_feats, stats):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.numeric_feats = numeric_feats
        self.stats = stats
        self.tab_dim = len(numeric_feats)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        hb  = float(row[HB_COL])
        lbl = 0 if hb < HB_THRESH else 1
        img = Image.open(row["_img_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        tab = torch.from_numpy(_row_tab_vector(row, self.numeric_feats, self.stats))
        if self.tab_dim == 0:                      # image-only ablation
            tab = torch.zeros(0, dtype=torch.float32)
        return img, tab, torch.tensor(lbl, dtype=torch.long), torch.tensor(hb, dtype=torch.float32)

print("Tabular + MMDataset ready.")


## Section 7 — Fusion wrapper (pre-hook on the backbone's terminal head)

In [ ]:
def _get_head(image_model):
    """Terminal head the pre-hook attaches to. reg_head is the head reported here."""
    for attr in ("reg_head", "cls_head"):
        h = getattr(image_model, attr, None)
        if h is not None:
            return attr, h
    raise AttributeError("backbone exposes neither reg_head nor cls_head")

def _infer_feat_dim(cls_head):
    if isinstance(cls_head, nn.Linear):
        return cls_head.in_features
    for m in cls_head.modules():
        if isinstance(m, nn.Linear):   return m.in_features
        if isinstance(m, nn.LayerNorm):
            ns = m.normalized_shape
            return ns[0] if isinstance(ns, (tuple, list)) else int(ns)
    raise ValueError("Could not infer feat_dim from cls_head")

def _mlp(inp, hid):
    return nn.Sequential(nn.Linear(inp, hid), nn.GELU(),
                         nn.LayerNorm(hid), nn.Linear(hid, hid), nn.GELU())

def _make_heads(dim, dropout):
    cls = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, 2))
    reg = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, max(dim // 2, 8)),
                        nn.GELU(), nn.Linear(max(dim // 2, 8), 1))
    return cls, reg

class MultiModalWrapper(nn.Module):
    def __init__(self, image_model, tab_dim, fusion="film",
                 tab_hidden=TAB_HIDDEN, fuse_dim=FUSE_DIM, dropout=TAB_DROPOUT):
        super().__init__()
        assert tab_dim > 0, "MultiModalWrapper needs tab_dim>0 (use ImageOnlyWrapper otherwise)"
        self.image_model = image_model
        self.fusion = fusion
        self.tab_dim = tab_dim
        self.head_name, _head = _get_head(image_model)
        feat_dim = _infer_feat_dim(_head)
        self.feat_dim = feat_dim
        self._feat = None
        _head.register_forward_pre_hook(self._hook)

        if fusion == "concat":
            self.tab_enc = _mlp(tab_dim, tab_hidden); fused = feat_dim + tab_hidden
        elif fusion == "film":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.film = nn.Linear(tab_hidden, feat_dim * 2); fused = feat_dim
        elif fusion == "gate":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.gate = nn.Linear(tab_hidden, feat_dim); fused = feat_dim
        elif fusion == "attn":
            self.img_proj = nn.Linear(feat_dim, fuse_dim)
            self.tok_proj = nn.ModuleList([nn.Linear(1, fuse_dim) for _ in range(tab_dim)])
            self.attn = nn.TransformerEncoderLayer(
                d_model=fuse_dim, nhead=4, dim_feedforward=fuse_dim * 2,
                dropout=dropout, batch_first=True, activation="gelu")
            fused = fuse_dim
        else:
            raise ValueError(f"Unknown fusion '{fusion}'")
        self.fused_dim = fused
        self.cls_head, self.reg_head = _make_heads(fused, dropout)

    def _hook(self, module, inp): self._feat = inp[0]

    def forward(self, img, tab):
        _ = self.image_model(img)               # runs backbone; hook captures pooled feat
        f = self._feat                           # (B, feat_dim)
        tab = tab.to(f.dtype)
        if self.fusion == "concat":
            fused = torch.cat([f, self.tab_enc(tab)], dim=-1)
        elif self.fusion == "film":
            g, b = self.film(self.tab_enc(tab)).chunk(2, dim=-1)
            fused = f * (1.0 + g) + b
        elif self.fusion == "gate":
            fused = f * torch.sigmoid(self.gate(self.tab_enc(tab)))
        elif self.fusion == "attn":
            toks = [self.img_proj(f).unsqueeze(1)]
            for i, proj in enumerate(self.tok_proj):
                toks.append(proj(tab[:, i:i+1]).unsqueeze(1))
            seq = torch.cat(toks, dim=1)         # (B, 1+tab_dim, fuse_dim)
            fused = self.attn(seq)[:, 0]         # image-token output
        return self.cls_head(fused), self.reg_head(fused).squeeze(-1)

class ImageOnlyWrapper(nn.Module):
    """Baseline: ignores tabular input, keeps the model's own heads. Unified signature."""
    def __init__(self, image_model):
        super().__init__(); self.image_model = image_model
    def forward(self, img, tab):
        return self.image_model(img)

class TabularOnlyModel(nn.Module):
    """Reference: predicts from tabular variables alone (no image)."""
    def __init__(self, tab_dim, hidden=64, dropout=TAB_DROPOUT):
        super().__init__()
        self.net = _mlp(max(tab_dim, 1), hidden)
        self.tab_dim = tab_dim
        self.cls_head, self.reg_head = _make_heads(hidden, dropout)
    def forward(self, img, tab):
        if self.tab_dim == 0:
            tab = torch.zeros(img.size(0), 1, device=img.device)
        h = self.net(tab)
        return self.cls_head(h), self.reg_head(h).squeeze(-1)

print("Fusion wrappers ready:  MultiModalWrapper / ImageOnlyWrapper / TabularOnlyModel")


## Section 8 — Losses and metrics

In [ ]:
# ---------------- CLASSIFICATION ----------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def build_cls_loss(train_labels):
    wt = None
    if CLS_CONFIG["use_class_weights"] and len(train_labels) > 0:
        w = compute_class_weight("balanced", classes=np.array([0, 1]),
                                 y=np.array(train_labels))
        wt = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    if CLS_CONFIG["use_focal_loss"]:
        return FocalLoss(gamma=CLS_CONFIG["focal_gamma"], weight=wt)
    return nn.CrossEntropyLoss(weight=wt)

def cls_metrics(labels, preds, scores):
    m = {"acc": accuracy_score(labels, preds),
         "f1":  f1_score(labels, preds, average="macro", zero_division=0)}
    try:    m["auc"] = roc_auc_score(labels, scores)
    except Exception: m["auc"] = float("nan")
    return m

def find_best_threshold(labels, scores):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.01):
        f = f1_score(labels, [1 if s >= t else 0 for s in scores],
                     average="macro", zero_division=0)
        if f > best_f: best_f, best_t = f, round(float(t), 2)
    return best_t, best_f

# ---------------- REGRESSION ----------------
def _norm_hb(hb):   return (hb - REG_CONFIG["hb_min"]) / (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"])
def _denorm_hb(x):  return x * (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"]) + REG_CONFIG["hb_min"]

def build_reg_loss():
    fn = REG_CONFIG["loss_fn"].lower()
    if fn == "huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
    if fn == "mae":   return nn.L1Loss()
    return nn.MSELoss()

def reg_metrics(hb_true, hb_pred):
    at, ap = np.asarray(hb_true), np.asarray(hb_pred)
    return {"mae":  float(np.mean(np.abs(at - ap))),
            "rmse": float(np.sqrt(np.mean((at - ap) ** 2))),
            "r2":   float(r2_score(at, ap)) if len(at) > 1 else float("nan")}

class EarlyStopper:
    def __init__(self, patience): self.p = patience; self.n = 0; self.best = float("inf")
    def step(self, s):
        if s < self.best - 1e-6: self.best = s; self.n = 0; return False
        self.n += 1; return self.p > 0 and self.n >= self.p

print("Losses & metrics ready (classification and regression kept separate).")


## Section 9 — Epoch loop

In [ ]:
def run_epoch_mm(model, loader, cls_loss=None, reg_loss=None,
                 optimizer=None, grad_accum=1, scaler=None):
    import contextlib
    training = optimizer is not None
    model.train() if training else model.eval()
    total = 0.0
    P, Y, S, HBP, HBT = [], [], [], [], []
    if training: optimizer.zero_grad()
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi, (imgs, tab, labels, hb) in enumerate(
                tqdm(loader, leave=False, desc="train" if training else "eval ")):
            imgs, tab = imgs.to(DEVICE), tab.to(DEVICE)
            labels, hb = labels.to(DEVICE), hb.to(DEVICE)
            amp = torch.cuda.amp.autocast() if scaler is not None else contextlib.nullcontext()
            with amp:
                logits, hb_pred = model(imgs, tab)
                if TASK == "classification":
                    loss = cls_loss(logits, labels)
                else:
                    ht = _norm_hb(hb) if REG_CONFIG["normalize_hb"] else hb
                    loss = reg_loss(hb_pred.view(-1), ht.view(-1))
            if training:
                (scaler.scale(loss / grad_accum) if scaler else (loss / grad_accum)).backward()
                last = (bi + 1 == len(loader))
                if (bi + 1) % grad_accum == 0 or last:
                    if scaler:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer); scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    optimizer.zero_grad()
            total += loss.item()
            if TASK == "classification":
                probs = torch.softmax(logits, dim=1)
                P.extend(logits.argmax(1).cpu().tolist())
                Y.extend(labels.cpu().tolist())
                S.extend(probs[:, 1].cpu().tolist())
            else:
                hp = hb_pred.detach().view(-1)
                if REG_CONFIG["normalize_hb"]: hp = _denorm_hb(hp)
                HBP.extend(hp.cpu().tolist()); HBT.extend(hb.cpu().view(-1).tolist())
    metrics = {"loss": total / max(len(loader), 1)}
    if TASK == "classification": metrics.update(cls_metrics(Y, P, S))
    else:                        metrics.update(reg_metrics(HBT, HBP))
    return metrics, Y, P, HBT, HBP, S

print("run_epoch_mm ready.")


## Section 10 — K-fold runner

In [ ]:
RESULTS = []          # main-pass results
ABLATION_RESULTS = []  # ablation results

def _fold_indices():
    if K == 1:
        idx = np.arange(len(POOL_DF))
        tr, vl = train_test_split(idx, test_size=VAL_SPLIT,
                                  random_state=SEED, stratify=POOL_LBL)
        return [(tr, vl)]
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    return list(skf.split(POOL_DF, POOL_LBL))

def _build_loaders(tr_df, vl_df, numeric_feats, stats):
    T_TR, T_VA = build_transforms(True), build_transforms(False)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), drop_last=False)
    tr_ds = MMDataset(tr_df, T_TR, numeric_feats, stats)
    vl_ld = DataLoader(MMDataset(vl_df, T_VA, numeric_feats, stats),
                       shuffle=False, **kw)
    ts_ld = DataLoader(MMDataset(TEST_DF, T_VA, numeric_feats, stats),
                       shuffle=False, **kw)
    if USE_AUGMENTATION:
        tr_ld = make_balanced_loader(
            dataset=tr_ds, hb_values=tr_df[HB_COL].values, batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS, device=DEVICE, hb_filter_min=HB_FILTER_MIN,
            hb_filter_max=HB_FILTER_MAX, aug_bins=AUG_BINS)
    else:
        tr_ld = DataLoader(tr_ds, shuffle=True, **kw)
    return tr_ld, vl_ld, ts_ld

def _make_sched(opt):
    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
        w = LinearLR(opt, start_factor=0.05, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        c = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        return SequentialLR(opt, [w, c], milestones=[WARMUP_EPOCHS])
    if SCHEDULER == "cosine":  return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    if SCHEDULER == "plateau": return torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    return None

def run_model_kfold_mm(name, model_factory, numeric_feats,
                       results_list=None, tag=""):
    """Train `name` with CV. model_factory(tab_dim)->model with forward(img,tab)."""
    if results_list is None: results_list = RESULTS
    tab_dim = len(numeric_feats)
    folds = _fold_indices()
    sep = "=" * 72
    print(f"\n{sep}\n  {name}   [{('holdout' if K==1 else str(K)+'-fold')}]"
          f"   feats={numeric_feats}  {tag}\n{sep}")

    accs, aucs, f1s, maes, rmses, r2s = [], [], [], [], [], []
    ts_scores, ts_preds, ts_hbp = [], [], []
    ts_y = ts_hbt = None
    t0 = time.time(); params = 0

    for fi, (tr_idx, vl_idx) in enumerate(folds):
        tr_df = POOL_DF.iloc[tr_idx]; vl_df = POOL_DF.iloc[vl_idx]
        stats = fit_tab_stats(tr_df, numeric_feats)
        tr_ld, vl_ld, ts_ld = _build_loaders(tr_df, vl_df, numeric_feats, stats)

        cls_loss = build_cls_loss((tr_df[HB_COL] < HB_THRESH).astype(int).tolist())
        reg_loss = build_reg_loss()

        model = model_factory(tab_dim).to(DEVICE)
        if params == 0:
            params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        sch = _make_sched(opt)
        scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None
        stop = EarlyStopper(EARLY_STOP)

        for ep in range(1, EPOCHS + 1):
            tr, *_ = run_epoch_mm(model, tr_ld, cls_loss, reg_loss, opt, GRAD_ACCUM, scaler)
            vl, *_ = run_epoch_mm(model, vl_ld, cls_loss, reg_loss)
            if sch is not None:
                sch.step(vl["loss"]) if SCHEDULER == "plateau" else sch.step()
            line = f"    F{fi+1} Ep{ep:02d}/{EPOCHS} TrL {tr['loss']:.4f} VlL {vl['loss']:.4f}"
            if TASK == "classification":
                line += f"  Acc {vl['acc']:.3f} AUC {vl['auc']:.3f} F1 {vl['f1']:.3f}"
            else:
                line += f"  MAE {vl['mae']:.3f} RMSE {vl['rmse']:.3f}"
            print(line)
            if stop.step(vl["loss"]):
                print(f"    early stop @ ep{ep}"); break

        ts, y, p, hbt, hbp, sc = run_epoch_mm(model, ts_ld, cls_loss, reg_loss)
        if ts_y is None: ts_y, ts_hbt = y, hbt
        ts_scores.append(sc); ts_preds.append(p); ts_hbp.append(hbp)
        if TASK == "classification":
            accs.append(ts["acc"]); aucs.append(ts["auc"]); f1s.append(ts["f1"])
            print(f"    F{fi+1} TEST  Acc {ts['acc']:.4f}  AUC {ts['auc']:.4f}  F1 {ts['f1']:.4f}")
        else:
            maes.append(ts["mae"]); rmses.append(ts["rmse"]); r2s.append(ts["r2"])
            print(f"    F{fi+1} TEST  MAE {ts['mae']:.4f}  RMSE {ts['rmse']:.4f}  R2 {ts['r2']:.4f}")
        del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
        gc.collect()

    elapsed = time.time() - t0
    print("  " + "-" * 68)
    row = dict(name=name, tag=tag, status="PASS", params=params, time_s=elapsed,
               folds=K, numeric_feats=list(numeric_feats))
    if TASK == "classification":
        row.update(acc_mean=float(np.nanmean(accs)), acc_std=float(np.nanstd(accs)),
                   auc_mean=float(np.nanmean(aucs)), auc_std=float(np.nanstd(aucs)),
                   f1_mean=float(np.nanmean(f1s)),  f1_std=float(np.nanstd(f1s)))
        print(f"  {name}:  Acc {row['acc_mean']:.4f}+/-{row['acc_std']:.4f}   "
              f"AUC {row['auc_mean']:.4f}+/-{row['auc_std']:.4f}   "
              f"F1 {row['f1_mean']:.4f}+/-{row['f1_std']:.4f}")
    else:
        row.update(mae_mean=float(np.nanmean(maes)), mae_std=float(np.nanstd(maes)),
                   rmse_mean=float(np.nanmean(rmses)), rmse_std=float(np.nanstd(rmses)),
                   r2_mean=float(np.nanmean(r2s)),   r2_std=float(np.nanstd(r2s)))
        print(f"  {name}:  MAE {row['mae_mean']:.4f}+/-{row['mae_std']:.4f}   "
              f"RMSE {row['rmse_mean']:.4f}+/-{row['rmse_std']:.4f}   "
              f"R2 {row['r2_mean']:.4f}+/-{row['r2_std']:.4f}")
    results_list.append(row)
    print(f"  done in {elapsed:.0f}s")
    return row

def _fail(name, e, results_list=None):
    if results_list is None: results_list = RESULTS
    print(f"  {name} FAILED: {e}"); traceback.print_exc()
    results_list.append(dict(name=name, status="FAIL", error=str(e)))

print("K-fold multimodal runner ready.")


## Section 11 — Backbone registry

In [ ]:
def _add(folder):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load(mod_name, folder, filename):
    _add(folder)
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(MODELS_DIR, folder, filename))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

def _register():
    reg = {}
    def safe(name, fn): reg[name] = fn
    # ---- Mamba variants ----
    try:
        m = _load("_m1", "01_Mamba_Official", "eye_hb_model.py")
        safe("Mamba1", lambda m=m: m.build_mamba1(IMG_SIZE, embed_dim=128, depth=4))
        safe("Mamba2", lambda m=m: m.build_mamba2(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e: print("  [skip] Mamba1/2:", e)
    try:
        m = _load("_m3", "06_Mamba3_Minimal", "eye_hb_model.py")
        safe("Mamba3", lambda m=m: m.build_model(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e: print("  [skip] Mamba3:", e)
    for key, folder in [("VMamba", "02_VMamba"), ("MambaVision", "03_MambaVision"),
                        ("MedMamba", "04_MedMamba"), ("VSSD", "05_VSSD_Mamba2Vision"),
                        ("DSA_Mamba", "07_DSA_Mamba_Custom")]:
        try:
            m = _load("_" + key, folder, "eye_hb_model.py")
            safe(key, lambda m=m: m.build_model(IMG_SIZE))
        except Exception as e: print(f"  [skip] {key}:", e)
    try:
        m = _load("_em", "08_EfficientMamba", "eye_hb_model.py")
        safe("EfficientMamba", lambda m=m: m.build_model(
            img_size=IMG_SIZE, embed_dim=256, depth=4, freeze_backbone=FREEZE_BACKBONE))
    except Exception as e: print("  [skip] EfficientMamba:", e)
    try:
        m = _load("_asm", "09_AdaptiveScanMamba", "adaptive_scan_mamba.py")
        safe("AdaptiveScan", lambda m=m: m.build_adaptive_scan_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"],
            colour_proj=NEW_MODEL_CFG["colour_proj"]))
    except Exception as e: print("  [skip] AdaptiveScan:", e)
    try:
        m = _load("_moe", "10_MoEMamba", "moe_mamba.py")
        safe("MoEMamba", lambda m=m: m.build_moe_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], n_experts=NEW_MODEL_CFG["n_experts"],
            d_state=NEW_MODEL_CFG["d_state"]))
    except Exception as e: print("  [skip] MoEMamba:", e)
    try:
        m = _load("_cf", "11_CrossFusionMamba", "cross_fusion_mamba.py")
        safe("CrossFusion", lambda m=m: m.build_cross_fusion_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"]))
    except Exception as e: print("  [skip] CrossFusion:", e)
    # ---- CNN / ViT baselines (timm) ----
    try:
        import timm
        class TimmWrapper(nn.Module):
            def __init__(self, backbone_name, pretrained=True):
                super().__init__()
                self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
                fd = self.backbone.num_features
                self.cls_head = nn.Sequential(nn.LayerNorm(fd), nn.Linear(fd, 2))
                self.reg_head = nn.Sequential(nn.LayerNorm(fd), nn.Linear(fd, 1))
            def forward(self, x):
                f = self.backbone(x)
                return self.cls_head(f), self.reg_head(f).squeeze(-1)
        for key, bb in [("ResNet50", "resnet50"), ("EfficientNetB3", "efficientnet_b3"),
                        ("DenseNet121", "densenet121"), ("MobileNetV3", "mobilenetv3_large_100"),
                        ("ViT_B16", "vit_base_patch16_224")]:
            safe(key, lambda bb=bb: TimmWrapper(bb, pretrained=True))
    except Exception as e: print("  [skip] timm baselines:", e)
    return reg

IMG_BUILDERS = _register()
print("Registered image builders:", list(IMG_BUILDERS.keys()))


## Section 12 — Reporting helpers

In [ ]:
import matplotlib.pyplot as plt

def results_frame(rows):
    """Tidy a results list into the table layout used in the paper."""
    ok = [r for r in rows if r.get("status") == "PASS"]
    if not ok:
        print("No successful runs."); return None
    df = pd.DataFrame(ok).sort_values("mae_mean", ascending=True)
    cols = ["name", "tag", "params", "mae_mean", "mae_std",
            "rmse_mean", "rmse_std", "r2_mean", "r2_std", "time_s"]
    return df[[c for c in cols if c in df.columns]].reset_index(drop=True)

def bar_chart(df, title, fname):
    if df is None or not len(df): return
    label = df["tag"] if df["name"].nunique() == 1 else df["name"]
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.45 * len(df))))
    ax.barh(label, df["mae_mean"], xerr=df["mae_std"], color="indianred")
    ax.set_xlabel("Test MAE g/dL (lower is better)")
    ax.invert_yaxis(); ax.set_title(title)
    plt.tight_layout(); plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()

print("Reporting helpers ready.")

## Stage 1 — Backbone comparison

All fourteen backbones with the default fusion and the three tabular variables. The
strongest extractor is carried into Stage 2.

In [ ]:
def make_factory(img_build_fn, use_tabular, fusion):
    if use_tabular:
        return lambda tab_dim: MultiModalWrapper(img_build_fn(), tab_dim, fusion=fusion)
    return lambda tab_dim: ImageOnlyWrapper(img_build_fn())

STAGE1 = []
for name, on in RUN.items():
    if not on:
        print(f"{name} skipped."); continue
    if name not in IMG_BUILDERS:
        print(f"{name}: no builder available, skipped."); continue
    try:
        factory = make_factory(IMG_BUILDERS[name], USE_TABULAR, FUSION)
        run_model_kfold_mm(name, factory, NUMERIC_FEATURES,
                           results_list=STAGE1, tag=f"fusion={FUSION}")
    except Exception as e:
        _fail(name, e, results_list=STAGE1)

S1 = results_frame(STAGE1)
if S1 is not None:
    S1 = S1.sort_values("mae_mean")
    print("\n=== Stage 1: fourteen backbones ranked by test MAE ===")
    print(S1.to_string(index=False))
    S1.to_csv("multihb_stage1_backbones.csv", index=False)
    bar_chart(S1, "Stage 1: fourteen backbones", "multihb_stage1_backbones.png")
    BEST_BACKBONE = S1.iloc[0]["name"]
    print(f"\nStrongest extractor: {BEST_BACKBONE}")
else:
    BEST_BACKBONE = ABLATION_MODEL

## Stage 2 — Fusion mechanism

The Stage 1 backbone is fixed and the four fusion operators are compared on the same
folds and the same split.

In [ ]:
STAGE2 = []
_base = IMG_BUILDERS.get(BEST_BACKBONE)
if _base is None:
    print(f"[skip] {BEST_BACKBONE} unavailable.")
else:
    for fmode in FUSION_MODES:
        try:
            factory = make_factory(_base, True, fmode)
            run_model_kfold_mm(BEST_BACKBONE, factory, NUMERIC_FEATURES,
                               results_list=STAGE2, tag=fmode)
        except Exception as e:
            _fail(f"{BEST_BACKBONE}[{fmode}]", e, results_list=STAGE2)

S2 = results_frame(STAGE2)
if S2 is not None:
    S2 = S2.sort_values("mae_mean")
    print("\n=== Stage 2: fusion sweep on " + str(BEST_BACKBONE) + " ===")
    print(S2.to_string(index=False))
    S2.to_csv("multihb_stage2_fusion.csv", index=False)
    bar_chart(S2, f"Stage 2: fusion sweep on {BEST_BACKBONE}", "multihb_stage2_fusion.png")
    BEST_FUSION = S2.iloc[0]["tag"]
    print(f"\nStrongest fusion: {BEST_FUSION}")
else:
    BEST_FUSION = FUSION

## Stage 3 — Which clinical variables to fuse

Backbone and fusion fixed. Each variable is added on its own, then all three together.
The best configuration is the proposed model, MultiHb.

In [ ]:
STAGE3 = []
if _base is not None:
    for label, feats in ABLATION_STEPS:
        try:
            factory = make_factory(_base, bool(feats), BEST_FUSION)
            run_model_kfold_mm(BEST_BACKBONE, factory, feats,
                               results_list=STAGE3, tag=label)
        except Exception as e:
            _fail(f"{BEST_BACKBONE}[{label}]", e, results_list=STAGE3)

S3 = results_frame(STAGE3)
if S3 is not None:
    print("\n=== Stage 3: per-variable ablation on " + str(BEST_BACKBONE)
          + "/" + str(BEST_FUSION) + " ===")
    print(S3.to_string(index=False))
    S3.to_csv("multihb_stage3_variables.csv", index=False)
    bar_chart(S3, f"Stage 3: per-variable ablation on {BEST_BACKBONE}", "multihb_stage3_variables.png")
    _best = S3.sort_values("mae_mean").iloc[0]
    print(f"\nProposed model (MultiHb): {BEST_BACKBONE} / {BEST_FUSION} / {_best['tag']}"
          f"   MAE {_best['mae_mean']:.4f} +/- {_best['mae_std']:.4f}")

## Results summary

In [ ]:
# Consolidated table matching the paper
import pandas as _pd
frames = []
for tag, df in [("Stage 1 backbones", S1 if "S1" in globals() else None),
                ("Stage 2 fusion",    S2 if "S2" in globals() else None),
                ("Stage 3 variables", S3 if "S3" in globals() else None)]:
    if df is not None:
        d = df.copy(); d.insert(0, "stage", tag); frames.append(d)
if frames:
    ALL = _pd.concat(frames, ignore_index=True)
    ALL.to_csv("multihb_all_results.csv", index=False)
    cols = [c for c in ["stage","name","tag","mae_mean","mae_std","rmse_mean","rmse_std",
                        "params","time_s"] if c in ALL.columns]
    print(ALL[cols].to_string(index=False))
    print("\nWritten: multihb_stage1_backbones.csv, multihb_stage2_fusion.csv,")
    print("         multihb_stage3_variables.csv, multihb_all_results.csv")
else:
    print("No results to summarise; run the stage cells first.")